# NHLF time lapse on calibrated gels — nematic activity analysis

Analyses phase-contrast time-lapse frames of normal human lung fibroblasts on
polyacrylamide gels at 5, 10 and 23 kPa, read from Google Drive.

**What this measures.** A fixed image tells you how ordered the sheet is. A time
lapse tells you whether the nematic is *active*, through two observables that
should agree:

1. **Defect self-propulsion.** A `+1/2` defect has a polar axis and is driven
   along it by active stress; a `-1/2` defect is three-fold symmetric and has no
   such direction. `speed(+1/2) > speed(-1/2)` is a direct signature of activity.
2. **Coarsening arrest.** A passive nematic coarsens — defect density decays as a
   power law. An active one nucleates pairs continuously and the density
   plateaus at a level set by activity over elasticity.

Two independent estimates of one quantity. If both rise with stiffness, that is
hard to explain as a segmentation or drift artefact.

**Before you trust any velocity**, fill in `SECONDS_PER_FRAME` and
`MICRONS_PER_PIXEL` in the setup cell. Neither can be read from the image files:
the frame rate stored in an exported video is a playback rate, not the
acquisition interval, and using it silently rescales every speed. Tracking runs
in pixels and frames, so calibration is applied at the end and nothing has to be
recomputed when the numbers turn up.

## 1. Install and mount

In [ ]:
!pip install -q git+https://github.com/Danpc11/lung-nematic.git

from google.colab import drive
drive.mount('/content/drive')

## 2. Configure

`FRAME_DIRS` maps a stiffness in kPa to a Drive folder of frames. Frames are
read in sorted filename order, so zero-padded names (`t0001.png`) are required —
`t1, t2, ... t10` sorts `t10` before `t2` and silently scrambles time.

In [ ]:
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')

FRAME_DIRS = {
    5.0:  DRIVE / 'nhlf_timelapse' / '5kPa',
    10.0: DRIVE / 'nhlf_timelapse' / '10kPa',
    23.0: DRIVE / 'nhlf_timelapse' / '23kPa',
}

OUTPUT_DIR = DRIVE / 'nhlf_timelapse' / 'resultados'

# --- acquisition metadata: fill these in -----------------------------------
SECONDS_PER_FRAME = None    # e.g. 600.0 for a 10-minute interval
MICRONS_PER_PIXEL = None    # of the time-lapse objective, NOT the histology one

# --- analysis parameters ---------------------------------------------------
# Gate for linking defects between frames. Set it a few times the expected
# per-frame displacement and well below the typical defect separation: too
# tight and real tracks break, too loose and an annihilation plus a nucleation
# get linked into one implausibly fast defect.
MAX_DISPLACEMENT_PX = 25.0

MAX_FRAMES = None           # cap for a quick trial run; None uses all

for stiffness, folder in FRAME_DIRS.items():
    print(f"{stiffness:5.1f} kPa  {'OK ' if folder.exists() else 'MISSING'}  {folder}")

## 3. Audit the frames

Check this before analysing anything. Mixed image sizes, an `RGB` mode where the
microscope should have produced `L` or `I;16`, or a frame count that does not
match the experiment are all worth resolving first.

In [ ]:
from PIL import Image

SUPPORTED = {'.png', '.tif', '.tiff', '.jpg', '.jpeg', '.bmp'}

def frame_paths(folder):
    return sorted(p for p in folder.iterdir()
                  if p.is_file() and p.suffix.lower() in SUPPORTED)

inventory = {}
for stiffness, folder in FRAME_DIRS.items():
    if not folder.exists():
        continue
    paths = frame_paths(folder)
    inventory[stiffness] = paths
    sizes, modes = set(), set()
    for path in paths[:20]:
        with Image.open(path) as image:
            sizes.add(image.size)
            modes.add(image.mode)
    print(f"{stiffness:5.1f} kPa: {len(paths):4d} frames | sizes {sizes} | modes {modes}")
    print(f"           first: {paths[0].name}   last: {paths[-1].name}")

if any(len(v) < 3 for v in inventory.values()):
    print("\n! a series with fewer than 3 frames cannot give kinetics")

## 3b. Choose the detection scale — do not skip this

The packaged `sigmas_px` default is `(40, 55, 70, 85)`, tuned for histology at
0.1147 um/px. On a time-lapse frame those kernels smooth the texture away and
**every frame returns zero defects** — no error, no warning, just empty tables
downstream. On a 300 px synthetic frame carrying a known +1/2 defect, the
default found nothing while `(12, 18)` recovered it.

Sigma should span roughly the cell-to-cell alignment length. The sweep below
tries several and reports what each finds; pick the smallest range that gives a
stable, non-zero count across frames. Too small tracks single-cell noise, too
large erases real defects.

In [ ]:
from dataclasses import replace

from lung_nematic.config import load_default_config
from lung_nematic.io_utils import read_rgb
from lung_nematic.phase_contrast import analyze_phase_contrast

CANDIDATE_SIGMAS = [(8., 12.), (12., 18.), (18., 26.), (26., 38.), (40., 55.)]

probe_stiffness = sorted(inventory)[0]
probe_paths = inventory[probe_stiffness][:5]

print(f"probing on {probe_stiffness:g} kPa, {len(probe_paths)} frames\n")
print(f"{'sigmas_px':>16}  {'defects per frame':>24}  {'mean coverage':>14}")
for sigmas in CANDIDATE_SIGMAS:
    trial = replace(load_default_config(), sigmas_px=sigmas)
    counts, coverage = [], []
    for path in probe_paths:
        result = analyze_phase_contrast(read_rgb(path), trial)
        counts.append(len(result['defects']))
        coverage.append(result['coverage_fraction'])
    print(f"{str(sigmas):>16}  {str(counts):>24}  "
          f"{sum(coverage) / len(coverage):>14.2f}")

# Set this from the sweep above before running the next cell.
SIGMAS_PX = (12.0, 18.0)

## 4. Per-frame nematic analysis

`analyze_phase_contrast` handles one frame: illumination flattening, cell-texture
masking, the director field at every configured scale, and defects that persist
across scales. This is the slow cell — expect a few seconds per frame.

In [ ]:
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from dataclasses import replace

from lung_nematic.config import load_default_config
from lung_nematic.io_utils import read_rgb
from lung_nematic.phase_contrast import analyze_phase_contrast

config = replace(load_default_config(), sigmas_px=SIGMAS_PX)

# analyze_phase_contrast returns the scalar metrics at the top level of its
# dict, alongside the array outputs. Select by exclusion so a future metric is
# picked up automatically rather than silently dropped.
ARRAY_KEYS = {'field', 'mask', 'coverage_mask', 'defects'}

per_frame, defects_by_series = {}, {}

for stiffness, paths in inventory.items():
    if MAX_FRAMES:
        paths = paths[:MAX_FRAMES]

    rows, frames = [], []
    for index, path in enumerate(tqdm(paths, desc=f"{stiffness:g} kPa")):
        image = read_rgb(path)
        result = analyze_phase_contrast(image, config)

        summary = {k: v for k, v in result.items() if k not in ARRAY_KEYS}
        summary.update(frame=index, filename=path.name,
                       stiffness_kPa=stiffness,
                       height_px=image.shape[0], width_px=image.shape[1])
        rows.append(summary)

        found = result['defects']
        frames.append(
            found[['x_px', 'y_px', 'charge']].copy()
            if len(found) else
            pd.DataFrame(columns=['x_px', 'y_px', 'charge'], dtype=float)
        )

    table = pd.DataFrame(rows)
    per_frame[stiffness] = table
    defects_by_series[stiffness] = frames

    # A correlation length that is negative or larger than the frame means the
    # exponential fit did not converge - usually a near-perfectly ordered field
    # where the correlation never decays. The value is meaningless, not small.
    bad = ((table.correlation_length_px <= 0)
           | (table.correlation_length_px > table.width_px)).sum()
    print(f"{stiffness:g} kPa: {sum(len(f) for f in frames)} detections over "
          f"{len(frames)} frames | mean coverage "
          f"{table.coverage_fraction.mean():.2f} | "
          f"{bad} frames with an unusable correlation length")

## 5. Track defects and remove drift

Linking is solved per charge class, so charge is conserved structurally rather
than filtered afterwards.

Drift is not a common offset that cancels. It adds a common velocity *vector*,
and speed is a magnitude, so with independent propulsion directions
`|v_prop + v_drift| != |v_prop| + |v_drift|` — drift inflates both classes *and*
compresses the contrast between them. On synthetic frames, 2 px/frame of drift
against 3 px/frame of propulsion shrinks the contrast by 41%. Subtract it.

The estimator is the median step over all tracked defects, which is only valid
when defect motion is otherwise uncorrelated. If the `+1/2` and `-1/2` medians
printed below differ substantially, the common component is not pure drift —
the sheet is flowing — and subtracting it removes real signal.

In [ ]:
from lung_nematic.tracking import (
    defect_kinetics, estimate_drift, motility_by_charge,
    subtract_drift, track_defects, track_summary,
)
import numpy as np

tracks_by_series, drift_by_series = {}, {}

for stiffness, frames in defects_by_series.items():
    raw = track_defects(frames, max_displacement_px=MAX_DISPLACEMENT_PX)
    drift = estimate_drift(raw)

    for charge in sorted(raw.charge.unique()):
        subset = estimate_drift(raw[raw.charge == charge])
        if not subset.empty:
            print(f"{stiffness:5g} kPa  charge {charge:+.1f}  median step "
                  f"({subset.drift_x_px.median():+.2f}, "
                  f"{subset.drift_y_px.median():+.2f}) px/frame")

    tracks_by_series[stiffness] = subtract_drift(raw, drift)
    drift_by_series[stiffness] = drift
    print(f"{stiffness:5g} kPa  {raw.track_id.nunique()} tracks, "
          f"cumulative drift ({drift.drift_x_px.sum():+.1f}, "
          f"{drift.drift_y_px.sum():+.1f}) px\n")

## 6. Signature 1 — defect self-propulsion

`speed(+1/2) > speed(-1/2)` says the nematic is active. The contrast between
them scales with activity, so it should rise with stiffness if substrate
rigidity drives contractility.

In [ ]:
motility = []
for stiffness, tracks in tracks_by_series.items():
    table = motility_by_charge(tracks)
    table['stiffness_kPa'] = stiffness
    motility.append(table)
motility = pd.concat(motility, ignore_index=True)

wide = motility.pivot_table(index='stiffness_kPa', columns='charge',
                            values='mean_speed_px_per_frame')
if 0.5 in wide.columns and -0.5 in wide.columns:
    wide['contrast'] = wide[0.5] - wide[-0.5]
    wide['ratio'] = wide[0.5] / wide[-0.5]

if SECONDS_PER_FRAME and MICRONS_PER_PIXEL:
    from lung_nematic.tracking import calibrate
    motility = calibrate(motility, MICRONS_PER_PIXEL, SECONDS_PER_FRAME)
    print(motility[['stiffness_kPa', 'charge', 'n_tracks', 'n_steps',
                    'mean_speed_um_per_s']].round(5).to_string(index=False))
else:
    print("! SECONDS_PER_FRAME / MICRONS_PER_PIXEL not set - "
          "speeds are px/frame, comparable across stiffness but not physical\n")
    print(motility[['stiffness_kPa', 'charge', 'n_tracks', 'n_steps',
                    'mean_speed_px_per_frame',
                    'sem_speed_px_per_frame']].round(3).to_string(index=False))

print()
print(wide.round(3).to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for charge, marker, label in [(0.5, 'o', '+1/2'), (-0.5, 's', '-1/2')]:
    subset = motility[motility.charge == charge].sort_values('stiffness_kPa')
    if subset.empty:
        continue
    axes[0].errorbar(subset.stiffness_kPa, subset.mean_speed_px_per_frame,
                     yerr=subset.sem_speed_px_per_frame, marker=marker,
                     capsize=3, label=label)
axes[0].set_xlabel('substrate stiffness (kPa)')
axes[0].set_ylabel('defect speed (px / frame)')
axes[0].set_title('Self-propulsion by charge class')
axes[0].legend(title='charge')

if 'contrast' in wide.columns:
    axes[1].plot(wide.index, wide.contrast, marker='D', color='crimson')
    axes[1].set_xlabel('substrate stiffness (kPa)')
    axes[1].set_ylabel('speed(+1/2) - speed(-1/2)   (px / frame)')
    axes[1].set_title('Activity contrast')
    axes[1].axhline(0, lw=0.8, color='0.6')

plt.tight_layout()
plt.show()

## 7. Signature 2 — coarsening arrest

A passive nematic coarsens: deaths outnumber births and the defect count decays.
An active steady state balances the two and the count plateaus.

Reading births against deaths is more direct than fitting a decay exponent,
because a plateau and a slow power law look alike over a short series but their
birth/death balance does not.

In [ ]:
kinetics = []
for stiffness, tracks in tracks_by_series.items():
    metrics = per_frame[stiffness]

    # There is no tissue area in a phase-contrast frame the way there is in
    # histology; the covered area has to be built from coverage_fraction and
    # the frame geometry, which needs the pixel size. Without it, work in
    # counts - they are still comparable across stiffness at equal field size.
    areas = None
    if MICRONS_PER_PIXEL:
        mm2 = (metrics.coverage_fraction * metrics.width_px * metrics.height_px
               * (MICRONS_PER_PIXEL / 1000.0) ** 2)
        areas = dict(zip(metrics.frame, mm2))

    table = defect_kinetics(tracks, areas)
    table['stiffness_kPa'] = stiffness
    kinetics.append(table)
kinetics = pd.concat(kinetics, ignore_index=True)

for stiffness, group in kinetics.groupby('stiffness_kPa'):
    half = len(group) // 2
    early, late = group.iloc[:half], group.iloc[half:]
    print(f"{stiffness:5g} kPa   first half: {early.n_defects.mean():6.1f} defects, "
          f"births {early.births.sum():3d} vs deaths {early.deaths.sum():3d}")
    print(f"{'':10s}   last half:  {late.n_defects.mean():6.1f} defects, "
          f"births {late.births.sum():3d} vs deaths {late.deaths.sum():3d}")

kinetics.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for stiffness, group in kinetics.groupby('stiffness_kPa'):
    axes[0].plot(group.frame, group.n_defects, label=f'{stiffness:g} kPa')
    axes[1].plot(group.frame, (group.births - group.deaths).cumsum(),
                 label=f'{stiffness:g} kPa')

axes[0].set_xlabel('frame')
axes[0].set_ylabel('defects in field')
axes[0].set_title('Density: decay or plateau?')
axes[0].legend()

axes[1].set_xlabel('frame')
axes[1].set_ylabel('cumulative births - deaths')
axes[1].set_title('Nucleation vs annihilation balance')
axes[1].axhline(0, lw=0.8, color='0.6')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Save

Everything is written in pixels and frames when the calibration is unset, so the
tables stay valid and can be rescaled later with
`lung_nematic.tracking.calibrate`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.concat(per_frame.values(), ignore_index=True).to_csv(
    OUTPUT_DIR / 'per_frame_metrics.tsv', sep='\t', index=False)
motility.to_csv(OUTPUT_DIR / 'motility_by_charge.tsv', sep='\t', index=False)
kinetics.to_csv(OUTPUT_DIR / 'defect_kinetics.tsv', sep='\t', index=False)

all_tracks = pd.concat(
    [t.assign(stiffness_kPa=s) for s, t in tracks_by_series.items()],
    ignore_index=True)
all_tracks.to_csv(OUTPUT_DIR / 'defect_tracks.tsv', sep='\t', index=False)

summaries = pd.concat(
    [track_summary(t).assign(stiffness_kPa=s)
     for s, t in tracks_by_series.items()], ignore_index=True)
summaries.to_csv(OUTPUT_DIR / 'track_summary.tsv', sep='\t', index=False)

print(f"written to {OUTPUT_DIR}")
for path in sorted(OUTPUT_DIR.glob('*.tsv')):
    print(f"  {path.name}")

## Reading the result

**Both signatures agree and rise with stiffness.** The strongest outcome: two
independent estimates of activity moving together is difficult to produce by
artefact. Report the contrast and the plateau level side by side.

**Self-propulsion is present but the density still decays.** The sheet is active
yet has not reached steady state within the recording. The plateau level is not
measurable here; report the contrast alone and say the series is too short.

**Neither signature appears.** Either the sheet is genuinely passive over this
window, or the tracking gate is wrong. Check `track_summary`: if median track
length is 1–2 frames, `MAX_DISPLACEMENT_PX` is too tight and real tracks are
being broken.

**`+1/2` and `-1/2` medians differed in the drift cell.** The common component
is not pure drift — the whole sheet is flowing. Subtracting it removed real
signal, and the speeds above understate activity.

A caution on the unit of analysis: `n_steps` counts every frame-to-frame
displacement, which is not an independent sample. Consecutive steps of one
defect are correlated, so the SEM above is optimistic. For inference across
stiffness, aggregate to one value per track — or better, per field of view — and
compare those.